<a href="https://colab.research.google.com/github/atenine/LING-230-Single-Model-Disfluency-Detector/blob/new_arch/scripts/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

login('')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Create destination folder first
os.makedirs('/content/data', exist_ok=True)

# Unpack — note the path after -C
!tar -xzf /content/drive/MyDrive/SEP-28k_CLIP.tar.gz -C /content/data/
print("Done unpacking!")

In [ ]:
csv_path   = '/content/SEP-28k-Extended_clips.csv'
clips_path = '/content/data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

In [ ]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


In [ ]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

In [ ]:
stutter_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

def create_soft_labels(row):
    return {
        'prolongation': row['Prolongation'] / 3.0,
        'block':        row['Block']        / 3.0,
        'soundrep':     row['SoundRep']     / 3.0,
        'wordrep':      row['WordRep']      / 3.0,
        'interjection': row['Interjection'] / 3.0,
        'fluent':       row['NoStutteredWords'] / 3.0
    }

soft_labels_df = df.apply(create_soft_labels, axis=1, result_type='expand')
df = pd.concat([df, soft_labels_df], axis=1)

print(df.head())

print("\nAverage probability per stutter type:")
for col in ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']:
    print(f"{col}: {df[col].mean():.3f}")

print("\nSample soft labels:")
print(df[['Show', 'EpId', 'ClipId',
          'prolongation', 'block',
          'soundrep', 'wordrep',
          'interjection', 'fluent']].head())

In [ ]:
subset_fraction = 0.1
train_df = df[df['SEP28k-T']== 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-T']== 'dev'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
test_df = df[df['SEP28k-T']== 'test'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# load voc2vec Encoder
print("\n Loading voc2vec encoder...")
MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

In [ ]:
def convo_audio_data(filename):
  try:
    waveform,sample_rate=torchaudio.load(filename)
    #ensures both input and encoder are on the same device
    waveform = waveform.to(device)
    # Stereo to mono
    if waveform.shape[0] > 1:
      waveform = torch.mean(waveform,dim=0,keepdim=True)
    # Resample
    if sample_rate != 16000:
      waveform = torchaudio.functional.sample(
          waveform,
          sample_rate,
          16000,
      )
    # extract features
    emission = encoder(waveform.to(device)).last_hidden_state
    emission = emission.detach().cpu().numpy()
    return emission
  except Exception as e:
    print(f"Error processing {filename}: {e}")
    return None

In [ ]:
# Hyper Parameters
batch_size    = 256
num_epochs    = 150
learning_rate = 0.0001

# Generate embeddings
def generate_embeddings(dataframe, desc):
    x         = []
    y         = []
    discarded = 0

    for _, row in tqdm(dataframe.iterrows(), desc=desc, total=len(dataframe)):
        embedding = convo_audio_data(row['filepath'])

        if embedding is None:
            discarded += 1
        elif ((np.shape(embedding)[0] != 1) |
              (np.shape(embedding)[1] != 149) |
              (np.shape(embedding)[2] != 768)):
            discarded += 1
        else:
            x.append(embedding)

            # Store soft labels
            y.append({
                'prolongation': torch.tensor(float(row['prolongation']), dtype=torch.float32),
                'block':        torch.tensor(float(row['block']),        dtype=torch.float32),
                'soundrep':     torch.tensor(float(row['soundrep']),     dtype=torch.float32),
                'wordrep':      torch.tensor(float(row['wordrep']),      dtype=torch.float32),
                'interjection': torch.tensor(float(row['interjection']), dtype=torch.float32),
            })

    print(f"Discarded in {desc}: {discarded}")
    return x, y


In [ ]:
# Dataset Class for Soft Labels
class AudioDataset(Dataset):
    def __init__(self, x, y, n_samples):
        self.x         = x
        self.y         = y
        self.n_samples = n_samples

    def __getitem__(self, index):
        # Convert embedding to tensor
        x_tensor = torch.tensor(self.x[index], dtype=torch.float32)
        return x_tensor, self.y[index]

    def __len__(self):
        return self.n_samples

# Generate embeddings
print("Generating embeddings")
data_start_time = datetime.now()

x_train, y_train = generate_embeddings(train_df, "Train")
x_valid, y_valid = generate_embeddings(valid_df, "Dev")
x_test,  y_test  = generate_embeddings(test_df,  "Test")

print("Time taken: " + str(datetime.now() - data_start_time))

# Create Datasets
n_samples_train = len(x_train)
n_samples_valid = len(x_valid)
n_samples_test  = len(x_test)

print(f'\nTrain samples: {n_samples_train}')
print(f'Dev samples:   {n_samples_valid}')
print(f'Test samples:  {n_samples_test}')

train_dataset = AudioDataset(x_train, y_train, n_samples_train)
valid_dataset = AudioDataset(x_valid, y_valid, n_samples_valid)
test_dataset  = AudioDataset(x_test,  y_test,  n_samples_test)

#Create DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("\nData loaders")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Model definition

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super(AttentionPooling, self).__init__()
        # Attention layer
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)                    # → (batch, time_steps, 1)
        weights = torch.softmax(scores, dim=1)         # → (batch, time_steps, 1)
        pooled  = torch.sum(x * weights, dim=1)        # → (batch, hidden_size)
        return pooled


class StutterNet(nn.Module):
    def __init__(self):
        super(StutterNet, self).__init__()

        # Layer 1 - Linear Projection
        # Reduces voc2vec output from 768 → 384
        self.projection = nn.Linear(768, 384)

        # Layer 2 - 1D CNN
        # Looks for local patterns in the audio
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=384, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 3 - 2 layers of BiLSTM
        # Looks at audio forward and backwards
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Layer 4 - Attention Pooling
        # Focuses on most important parts of audio
        self.attention = AttentionPooling(hidden_size=512)

        # Layer 5 - Dense Layers
        self.dense = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 6 - 5 Output Heads
        # One head per stutter type!
        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x shape coming in → (batch, 1, 149, 768)
        # Remove extra dimension
        x = x.squeeze(1)

        # Linear Projection
        x = self.projection(x)
        x = torch.relu(x)

        # 1D CNN
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        # BiLSTM
        x, _ = self.bilstm(x)

        # Attention Pooling
        x = self.attention(x)

        # Dense Layers
        x = self.dense(x)

        return {
            'prolongation': torch.sigmoid(self.head_prolongation(x).squeeze(1)),
            'block':        torch.sigmoid(self.head_block(x).squeeze(1)),
            'soundrep':     torch.sigmoid(self.head_soundrep(x)).squeeze(1),
            'wordrep':      torch.sigmoid(self.head_wordrep(x)).squeeze(1),
            'interjection': torch.sigmoid(self.head_interjection(x)).squeeze(1)
        }


# Initialize Model
model  = StutterNet().to(device)
print(model)

In [131]:
from torch.utils.tensorboard import SummaryWriter
# early_stopper = EarlyStopping(patience=5, delta=0.01)
writer = SummaryWriter()

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_accu= []
train_losses = []
eval_accu=[]
eval_losses=[]

classes = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']

def train(epoch):
        print('\nEpoch : %d'%epoch)

        model.train()

        running_loss = {head_name: 0 for head_name in classes}
        correct = {head_name: 0 for head_name in classes}
        total = {head_name: 0 for head_name in classes}

        for data in tqdm(train_loader, desc="Training", total=len(train_loader)):
            
            inputs, targets=data[0].to(device),data[1].to(device)
            # print(targets)
            # forward pass
            optimizer.zero_grad()
            outputs=model(inputs)
            
            loss = 0
            for head_name in classes:
                loss += criterion(outputs[head_name], targets)
                running_loss[head_name] += loss.item()
            loss = loss / 5  # Average loss across heads
            
            loss.backward(retain_graph=True)
            optimizer.step()

            # print(outputs)
            # Compute Training Accuracy
            for head_name in classes:
                predicted_labels = torch.round(outputs[head_name][:, 0])
                # print(predicted_labels)
                total[head_name] += targets.size(0)
                correct[head_name] += predicted_labels.eq(targets).sum().item()
        
        for head_name in classes:
            accu = 100.*correct[head_name] / total[head_name]
            train_loss = running_loss[head_name] / len(train_loader)
            # visualisation
            writer.add_scalar(head_name + " Loss/train", loss, epoch)  
            writer.add_scalar(head_name + " Accuracy/train", accu, epoch)  
            
            train_accu.append(accu)
            train_losses.append(train_loss)
            print(head_name + 'Loss:\t %.3f | Accuracy: %.3f'%(train_loss,accu))

            

def eval(epoch):
    model.eval()

    running_loss = {head_name: 0 for head_name in classes}
    correct = {head_name: 0 for head_name in classes}
    total = {head_name: 0 for head_name in classes}

    with torch.no_grad():
        for data in tqdm(valid_loader, desc="Validating", total=len(valid_loader)):
            inputs, targets = data[0].to(device),data[1].to(device)
            
            outputs = model(inputs)
            
            loss = 0
            for head_name in classes:
                loss += criterion(outputs[head_name], targets)
                running_loss[head_name] += loss.item()

            for head_name in classes:
                predicted_labels = torch.round(outputs[head_name][:, 0])
                # print(predicted_labels)
                total[head_name] += targets.size(0)
                correct[head_name] += predicted_labels.eq(targets).sum().item()
    
    for head_name in classes:
        validate_loss = running_loss[head_name] / len(valid_loader)

        accu=100.*correct[head_name] / total[head_name]
        
        # visualisation
        writer.add_scalar("Loss/test", loss, epoch)  
        writer.add_scalar("Accuracy/test", accu, epoch)  

        eval_accu.append(accu)
        eval_losses.append(validate_loss)

        print(head_name + ' Loss:\t %.3f | Accuracy: %.3f'%(validate_loss, accu))
    return validate_loss

In [ ]:
torch.save(model, '../models/DisfluencyNet_snd_quart_blocks.pth')

epochs=num_epochs
train_start_time = datetime.now()
for epoch in range(1,epochs+1):
    train(epoch)
    eval_loss = eval(epoch)
    torch.cuda.empty_cache()
    # if early_stopper(eval_loss, model):
    #     print('Eval performance not improving; would be early stopping now!')
    #     model.load_state_dict(early_stopper.best_model)
    #     # TODO: figure out why break makes our number of predictions go to zero :|
    #     # break

train_end_time = datetime.now()
print("\nFinished training model in " + str(train_end_time - train_start_time) + " seconds.\n")

##################################################################################################
# torch.save(model, 'DisfluencyNet_wp_quart.pth')
##################################################################################################
## TEST MODEL ##
test_loader = DataLoader(dataset=test_dataset,
                        batch_size=batch_size,
                        shuffle=True,
                        num_workers=0)

# model.load_state_dict(early_stopper.best_model)
model.eval()

with torch.no_grad():
    total = 0
    correct = 0
    n_correct = 0
    # Compute F1 score, precision and recall
    predicted_stutter = 0
    labels_stutter = 0
    correct_stutter = 0
    i = 0
    final_label = []
    final_predicted = []
    for features, labels in test_loader:
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        loss= criterion(outputs,labels)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        #    final_predicted.append(predicted)
        #    final_label.append(labels)

        # predicted = torch.reshape(predicted,(outputs.shape[0],1))

    for i in range (0, len(predicted)) :
        # F1 score for stutter
        if (predicted[i] == 1) :
            predicted_stutter +=1
        if (labels[i] == 1) :
            labels_stutter +=1
        if ((predicted[i] == 1) & (labels[i] == 1)):
            correct_stutter +=1
        if (predicted[i] == labels[i]) :
            n_correct = n_correct + 1

print(f'Predicted stutter: {predicted_stutter}%')
print(f'Correct stutter predictions: {correct_stutter}%')

acc_test = 100*correct/total
print(f'Accuracy of the network on test dataset is : {acc_test} %')
recall = correct_stutter/ labels_stutter
print(f'Recall of the network on test dataset is : {recall}')
precision = correct_stutter / predicted_stutter
print(f'Precision of the network on test dataset is : {precision}')
f1_score = 2 * precision * recall / (precision + recall)
print(f'F1 Score of the network on test dataset is : {f1_score}')